# Day 4 - Make It Safe to Run

The pipeline you built over the last three days runs. It produces output. Nobody raises an error.

That is not the same as it being safe to run.

`sales_raw.csv` has 30 rows. The cleaned table has 23. Seven rows disappeared and nothing told anyone. That is a **silent failure** - the pipeline produces output without crashing, the report lands in someone's inbox and looks fine.

This session makes those failures visible:

1. **Fail loudly** - check the raw (bronze) data and raise an error
2. **Collect** - find every problem in one run, not just the first

Stretch - each part builds on the last:

3. **Functions** - make your checks reusable
4. **Clean, then check again** - run the same checks on silver, plus some new ones
5. **Stop at the end** - raise once, with the full list
6. **Gold** - check the join before anyone reports on it

**There is no solution notebook for this activity.** Go as far as you can - the next activity does not depend on it.

---

## Setup

Run this cell first. It loads the raw sales file - **no cleaning yet**.

In [ ]:
import pandas as pd

df_raw = pd.read_csv('../HomeSphere/data/sales_raw.csv')

print(f'sales_raw: {len(df_raw)} rows')
df_raw.head(10)

---

# Part 1 - Fail loudly

Every check has three parts:

- **Risk** - what could go wrong
- **Check** - what you test for
- **If this fails** - what the pipeline should do

Use explicit logic that is readable and intentional:

```python
null_count = df_raw["product_id"].isnull().sum()
if null_count > 0:
    raise ValueError(f"{null_count} rows have no product_id. Pipeline stopped.")
```

`raise ValueError` stops the code and shows your message in red. Nobody can miss it.

## TODO 1 - Missing product

**Risk:** A sale with no `product_id` cannot be joined to the product catalogue. Its revenue belongs to no category.

**Check:** No nulls in `product_id`.

**If this fails:** Stop.

In [ ]:
null_count = df_raw["product_id"].isnull().sum()

# TODO: raise a clear ValueError if any product_id is missing

## TODO 2 - Duplicate rows

**Risk:** The same order appears twice. Revenue is counted twice.

**Check:** No fully duplicated rows.

**If this fails:** Stop.

In [ ]:
dup_count = df_raw.duplicated().sum()

# TODO: raise a clear ValueError if there are any duplicate rows

## TODO 3 - Quantity

**Risk:** A quantity that is not a number, or is zero, cannot be a real sale.

**Check:** Every quantity is a number greater than zero.

**If this fails:** Stop - and show which orders are affected.

`pd.to_numeric(..., errors='coerce')` turns anything that is not a number into `NaN`. `NaN > 0` is `False`, so `~(quantity > 0)` finds both problems at once.

In [ ]:
quantity = pd.to_numeric(df_raw["quantity"], errors="coerce")
bad_quantity = df_raw[~(quantity > 0)]

# TODO: raise a clear ValueError if bad_quantity is not empty
# hint: list(bad_quantity["order_id"]) gives the affected orders

## TODO 4 - Price

**Risk:** A zero or negative price would reduce revenue instead of adding to it.

**Check:** Every unit price is greater than zero.

**If this fails:** Stop - and show which orders are affected.

The prices need the `£` removing before they can be compared - that line is done for you.

In [ ]:
price = df_raw["unit_price"].astype(str).str.replace("£", "", regex=False).astype(float)
bad_price = df_raw[price <= 0]

# TODO: raise a clear ValueError if bad_price is not empty

Every one of your checks should fail. That is the point - the raw data has problems.

Now try **Run All** (or **Kernel > Restart & Run All**). The notebook stops at TODO 1. The other checks never run.

A pipeline works the same way. You would fix one problem, run it again, hit the next one, fix that, run it again...

**You only find out about one problem per run.**

---

# Part 2 - Collect every problem

Instead of raising straight away, add each problem to a list. Then print the whole list.

The first check is done for you. Add the other four.

In [ ]:
errors = []

# 1 - missing product_id
null_count = df_raw["product_id"].isnull().sum()
if null_count > 0:
    errors.append(f"{null_count} rows have no product_id")

# 2 - duplicate rows
# TODO


# 3 - quantity not a number, or zero or less
# TODO


# 4 - unit price zero or less
# TODO


# 5 - missing region
# TODO


for msg in errors:
    print(f"ERROR: {msg}")

print(f"{len(errors)} problems found")

Every problem, one run.

**Think:**

- The cleaning step drops rows with a missing `product_id`. It fills a missing `region` with `"Unknown"`. Is a missing region really an *error*, or just something to know about?
- Which of these problems would you want to stop the pipeline?

---

# Stretch Part 3 - Functions

Your checks follow the same few patterns. Turn them into functions so you can call them on any DataFrame - not just `df_raw`.

Each function takes the `errors` list and **appends** a message if the check fails. It does not raise, so you can call them all and still get the full list.

In [ ]:
def check_no_nulls(df, column, errors):
    # TODO: append a message to errors if column has any nulls
    pass


def check_no_duplicates(df, errors):
    # TODO: append a message to errors if df has any fully duplicated rows
    pass


def check_positive(df, column, errors):
    # TODO: append a message to errors if any value in column is not a number, or is zero or less
    # hint: pd.to_numeric(df[column], errors="coerce") turns anything that is not a number into NaN
    pass

Now call them on the raw data:

In [ ]:
bronze_errors = []

check_no_nulls(df_raw, "product_id", bronze_errors)
check_no_nulls(df_raw, "region", bronze_errors)
check_no_duplicates(df_raw, bronze_errors)
check_positive(df_raw, "quantity", bronze_errors)
check_positive(df_raw, "unit_price", bronze_errors)

for msg in bronze_errors:
    print(f"BRONZE: {msg}")

print(f"{len(bronze_errors)} problems found")

**Think:** `check_positive` on `unit_price` probably reports more problems than Part 2 did. Why? Is a price of `£129.99` really a problem - or something cleaning can fix?

---

# Stretch Part 4 - Clean, then check again

Bronze checks ask: **what is wrong with the data that arrived?** Failures there are expected - that is the information.

Silver checks ask a different question: **did cleaning fix it, without breaking anything?** A failure here means *your own code* is wrong.

Run the cleaning. This is the same logic you have used all week.

In [ ]:
df = df_raw.copy()
df['unit_price'] = df['unit_price'].astype(str).str.replace('£', '', regex=False).astype(float)
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=True, errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df = df.dropna(subset=['quantity'])
df['quantity'] = df['quantity'].astype(int)
df['status'] = df['status'].str.lower().str.strip()
df = df.drop_duplicates()
df = df.dropna(subset=['product_id'])
df['region'] = df['region'].fillna('Unknown')
df = df[(df['unit_price'] > 0) & (df['quantity'] > 0)]
silver_sales = df.reset_index(drop=True)

print(f'bronze: {len(df_raw)} rows')
print(f'silver: {len(silver_sales)} rows')

Now check silver. Two kinds of check:

**The same functions as bronze.** These should now pass - that proves the cleaning worked.

**New checks that only make sense after cleaning.** Cleaning can *create* problems:

- **Rows lost** - cleaning that drops too much is as bad as no cleaning. Fail if more than 25% of the rows were dropped.
- **Dates** - `errors='coerce'` quietly turns any date it cannot read into `NaT` (a missing date). Fail if any `order_date` is missing.
- **Status** - after cleaning, every status should be `complete`, `pending` or `cancelled`.

*Tip: the clean data should pass. To prove a check works, change it to make it fail - for example change 25% to 10% - then change it back.*

In [ ]:
silver_errors = []

# same functions, now on silver
# TODO: call your functions on silver_sales, appending to silver_errors


# rows lost
# TODO: append an error if more than 25% of the rows were dropped


# dates
# TODO: append an error if any order_date is missing (NaT)
# hint: .isnull() works on dates too


# status
# TODO: append an error if any status is not complete, pending or cancelled
# hint: ~silver_sales["status"].isin([...])


for msg in silver_errors:
    print(f"SILVER: {msg}")

print(f"{len(silver_errors)} problems found")

**Think:** `check_no_nulls` on `region` now passes. Did the problem go away - or did cleaning just hide it? How many rows say `"Unknown"`?

---

# Stretch Part 5 - Stop at the end

Everything above *prints*. Nothing stops.

On Day 2 you built a pipeline where **Track Failure** only runs if **Clean Sales Orders** fails. A notebook only fails when an error is raised. A notebook that prints its problems and finishes shows **Succeeded** - and Track Failure never fires.

That is a silent failure at the pipeline level.

So, finally, raise - once, at the end, with the whole list.

In [ ]:
# TODO: if silver_errors is not empty, raise a ValueError
# include how many problems were found in the message


print("All silver checks passed - safe to use")

**Think:** this only stops on silver errors. Should any of the bronze problems stop the pipeline too? Which ones?

---

# Stretch Part 6 - Gold

Gold is what the report is built from. Silver can pass every check and gold can still be wrong - because the **join** can go wrong.

Run this cell. It joins silver to the product catalogue and calculates `line_value`, the same as Day 1.


In [ ]:
import json

with open('../HomeSphere/data/products_raw.json') as f:
    products_data = json.load(f)

silver_products = pd.json_normalize(products_data['products'])[['product_id', 'name', 'category']]

# break it - uncomment ONE line at a time to see your checks fire
# silver_products = silver_products[silver_products['product_id'] != 'P007']    # a product missing from the catalogue
# silver_products = pd.concat([silver_products, silver_products.head(1)])        # a product listed twice

gold_revenue = silver_sales.merge(silver_products, on='product_id', how='left')
gold_revenue['line_value'] = gold_revenue['quantity'] * gold_revenue['unit_price']

print(f'silver_sales: {len(silver_sales)} rows')
print(f'gold_revenue: {len(gold_revenue)} rows')


Three checks:

- **Every sale found its product** - a `left` join keeps a sale even when its `product_id` is not in the catalogue. It just gets no `category`, and its revenue belongs to no category. Your `check_no_nulls` already does this.
- **Every line value is positive** - your `check_positive` already does this too.
- **The join did not add rows** - if a product appears twice in the catalogue, every sale of it appears twice in gold. Revenue is counted twice. Fail if `gold_revenue` has a different number of rows from `silver_sales`.

Finish with a `raise`, like Part 5.

*Tip: the clean data should pass. Uncomment one of the "break it" lines above, run both cells, then comment it out again.*


In [ ]:
gold_errors = []

# same functions, now on gold
# TODO: every row has a category, every line_value is positive


# rows added by the join
# TODO: append an error if gold_revenue and silver_sales have a different number of rows


for msg in gold_errors:
    print(f"GOLD: {msg}")

# TODO: if gold_errors is not empty, raise a ValueError


print("All gold checks passed - safe to report")


**Think:** a missing product and a duplicated product both come from the catalogue, not the sales file. Whose problem is it - and should it stop the sales pipeline?


---

## Debrief

The question is not *did your checks work?* It is: **what should the pipeline do when a check fails?**

- **Stop** - raise an error and halt. Nothing downstream runs.
- **Warn and continue** - report the problem and keep going. Risky if downstream code uses the bad data.
- **Quarantine** - set the failing rows aside, continue with the clean rows.

For each of your checks - which response makes most sense, and why? Does the answer change between bronze and silver?